# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')
print("Ready!", os.getcwd())

Cloning into 'flyrank-ML-internship'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 153 (delta 64), reused 107 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 1.85 MiB | 11.25 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Ready! /content/flyrank-ML-internship


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Task type: Scoring / Ranking
# No computation needed here — framing is explained in the markdown cell above.
print("Task type: Scoring / Ranking")

Task type: Scoring / Ranking


Task type: Scoring / Ranking

My lane is Query Intelligence — I am building a scoring system that ranks content-query pairs by optimization opportunity. This is a scoring and ranking task, not a simple classification, because the output is an ordered list of pages that deserve attention first — not just a yes/no label. The score combines query momentum signals (is this query gaining or losing clicks?) with page-level context (how visible is this page for this query?). A reviewer then acts on the top-ranked pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Target: is_rising = (clicks_last30 > clicks_prev30)")
print("Source: observed click counts — no product flags used")

Target: is_rising = (clicks_last30 > clicks_prev30)
Source: observed click counts — no product flags used


Target (proxy label):

I would predict whether a content-query pair is a rising opportunity — defined as: clicks from this query to this page were higher in the last 30 days than in the previous 30 days.

This is a proxy label, not a directly observed future outcome. It is computed from a defined rule on existing data:

is_rising = (clicks_last30 > clicks_prev30)

This proxy is safe because both windows are already in the past at decision time — there is no future information leaking in. It comes from an observed outcome (actual click counts), not from a product decision flag. The limitation is that it measures recent momentum, not guaranteed future growth.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Success metric: Precision@20")
print("Baseline to beat: simple impressions-first rule")
print("Target: Precision@20 > 0.60")

Success metric: Precision@20
Baseline to beat: simple impressions-first rule
Target: Precision@20 > 0.60


Success metric: Precision@20

My output is a ranked list of content-query pairs to review. The right metric is Precision@20 — of the top 20 pairs the system recommends, how many are genuinely rising opportunities?

A reviewer can realistically check 20 pairs per week. If the baseline (a simple rule like "highest impressions first") gets 8 of 20 right (Precision@20 = 0.40), my scoring system should beat that. A result above 0.60 would mean the model is meaningfully better than the rule.

I chose Precision@20 over accuracy because the list is used top-down — the first recommendation matters most, and a miss buried at rank 18 costs less than a miss at rank 1.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Show the unit of analysis — one row = one content item
lane_slice = df[['content_id', 'client_id', 'impressions_90d',
                  'clicks_90d', 'avg_position', 'ctr',
                  'trend_direction']].head(5)

print("One row = one content item (90-day window)")
print(f"Total rows: {len(df)}")
display(lane_slice)

One row = one content item (90-day window)
Total rows: 30000


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,down


One row = one content item paired with one query hash, over a fixed 90-day window.

The starter dataset approximates this — each row is one content item with its aggregated 90-day query signals.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Why ML beats a rule: interactions between position, CTR,")
print("impressions, trend, and content type are too complex for")
print("a single fixed threshold across 32 clients.")

Why ML beats a rule: interactions between position, CTR,
impressions, trend, and content type are too complex for
a single fixed threshold across 32 clients.


hy ML beats a fixed rule:

A fixed rule like "flag pages with impressions > 1000 and CTR < 0.5%" sounds simple — but it misses the complexity:

A page at position 9 with 0.4% CTR might be perfectly normal. A page at position 2 with 0.4% CTR is a serious problem. A fixed rule cannot tell the difference.
Query momentum shifts vary by client, content type, and season — no single threshold fits all 32 clients.
The combination of signals (position + CTR + impressions + trend + content age) interacts in ways a hand-written if-statement cannot capture without becoming impossibly complex.

ML learns these interactions from the data itself — finding which combinations actually predict rising or falling query performance — and ranks pages in a way no fixed rule can match.

And in the code cell below it paste:

python
print("Why ML beats a rule: interactions between position, CTR,")
print("impressions, trend, and content type are too complex for")
print("a single fixed threshold across 32 clients.")

Done?

The repo wasn't cloned in this session. Paste this at the top of your notebook as the very first code cell and run it first:

python
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')
print("Ready!", os.getcwd())

Run that first, then run the Section 4 code cell again. Done?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.